# 04 — Proposta B: localização e treino

Objetivo: executar localização B e documentar treino/inferência com entrada RGB [0,255]. Entradas: manifesto, anotações e configuração B. Saídas: candidatos, checkpoint e metadados quando o treino curto for solicitado. Pré-requisitos: ambiente do README. O modo padrão usa dublê sintético; `TCC_RUN_TRAINING_SMOKE=1` executa uma etapa com pesos aleatórios e persistência.

Execução: na raiz, `python -m jupyter lab notebooks`. Execute de cima para baixo. Variáveis: `TCC_ROOT`, `TCC_RUN_MODE=synthetic|real`, `TCC_MANIFEST`, `TCC_ANNOTATIONS`, `TCC_SPLIT`. Colab pode definir `TCC_ROOT` para a cópia do repositório e instalar as dependências em runtime Python compatível.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path(os.environ.get("TCC_ROOT", Path.cwd())).resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Defina TCC_ROOT para a raiz do repositório."
sys.path.insert(0, str(ROOT))
MODE = os.environ.get("TCC_RUN_MODE", "synthetic")
assert MODE in {"synthetic", "real"}
from scripts.notebook_support import prepare_synthetic, SyntheticClassifier
from src import data_protocol as dp
if MODE == "synthetic":
    OUTPUT, rows = prepare_synthetic(ROOT)
    MANIFEST = OUTPUT / "images.csv"
    ANNOTATIONS = OUTPUT / "synthetic_annotations.json"
else:
    OUTPUT = ROOT / "experiments" / "runs" / "notebook_real"
    OUTPUT.mkdir(parents=True, exist_ok=True)
    MANIFEST = Path(os.environ.get("TCC_MANIFEST", ROOT / "data/splits/lote_1/images.csv"))
    ANNOTATIONS = Path(os.environ.get("TCC_ANNOTATIONS", ROOT / "data/annotations/lote_1_pending.json"))
print({"modo": MODE, "raiz": str(ROOT), "manifesto": str(MANIFEST)})


In [ ]:
from src.train import load_config
from src.model_b import predict_image_b
cfg = load_config(ROOT / "configs/model_b.yaml")
if MODE == "synthetic":
    results = [predict_image_b(dp.read_image_rgb(row["path"]), SyntheticClassifier(), cfg, row["image_id"]) for row in rows]
    assert all(r["status"] != "erro" for r in results)
    print([(r["status"],len(r["candidates"])) for r in results])
else:
    from src.segmentation import propose_regions
    rows = dp.read_manifest(MANIFEST)
    image = dp.read_image_rgb(rows[0]["path"])
    candidates, diagnostics = propose_regions(image, cfg["localization"])
    print(diagnostics)
if os.environ.get("TCC_RUN_TRAINING_SMOKE", "0") == "1":
    subprocess.run([sys.executable, str(ROOT / "scripts/train_model_b.py"), "--smoke"], check=True, cwd=ROOT)
else:
    print("Treino curto não executado nesta célula. Use --smoke ou TCC_RUN_TRAINING_SMOKE=1.")


Os parâmetros de localização são provisórios. Muitos candidatos podem ser artefatos; a contagem não mede melhora. Treino real exige `--image-manifest`, `--crop-manifest` e `--annotations`, com duas classes revisadas em treino/validação. A CLI não seleciona pesos pelo teste. O artefato é `best.keras`, escolhido por `val_loss` e recarregado.